In [ ]:
# ============================================================
# OPTIMIZED SHAP (NO CRASH VERSION)
# ============================================================

import torch
import torch.nn as nn
import shap
import numpy as np
from PIL import Image
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# ------------------ FORCE CPU ------------------
device = torch.device("cpu")
print("Using device:", device)

# ============================================================
# 1. MODEL (USE YOUR EXACT CLASS)
# ============================================================

import torchvision.models as models

class CrossStitchUnit(nn.Module):
    def __init__(self):
        super().__init__()
        self.alpha = nn.Parameter(torch.tensor([[0.9, 0.1],
                                                [0.1, 0.9]], dtype=torch.float32))

    def forward(self, f1, f2):
        return (
            self.alpha[0,0]*f1 + self.alpha[0,1]*f2,
            self.alpha[1,0]*f1 + self.alpha[1,1]*f2
        )

class CrossStitchMultiTaskModel(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet18(weights=None)
        self.shared_layers = nn.Sequential(*list(backbone.children())[:-2])

        self.task1_conv = nn.Conv2d(512, 512, 3, padding=1)
        self.task2_conv = nn.Conv2d(512, 512, 3, padding=1)

        self.cross_stitch = CrossStitchUnit()
        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.maturity_head = nn.Linear(512, 3)
        self.disease_head = nn.Linear(512, 4)

    def forward(self, x):
        shared = self.shared_layers(x)

        f1 = self.task1_conv(shared)
        f2 = self.task2_conv(shared)

        f1, f2 = self.cross_stitch(f1, f2)

        f1 = self.pool(f1).view(f1.size(0), -1)
        f2 = self.pool(f2).view(f2.size(0), -1)

        return self.maturity_head(f1), self.disease_head(f2)

# ============================================================
# 2. LOAD MODEL
# ============================================================

model = CrossStitchMultiTaskModel()

state_dict = torch.load(
    "/Users/dhruviramaiya/Downloads/Mtech Major Project/Multitask/best_optimized_multitask_model.pth",
    map_location=device
)

model.load_state_dict(state_dict, strict=False)
model.eval()

# ============================================================
# 3. WRAPPERS (ONE TASK AT A TIME)
# ============================================================

class MaturityWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        out, _ = self.model(x)
        return out

# ONLY ONE TASK → prevents crash
wrapped_model = MaturityWrapper(model)

# ============================================================
# 4. IMAGE (LOW RES)
# ============================================================

transform = transforms.Compose([
    transforms.Resize((96, 96)),   # KEY CHANGE
    transforms.ToTensor(),
])

img_path = "test.jpg"
img = Image.open(img_path).convert("RGB")

input_tensor = transform(img).unsqueeze(0)

# Background = 1 image only
background = input_tensor.clone()

# ============================================================
# 5. SHAP EXPLAINER (LIGHTWEIGHT)
# ============================================================

explainer = shap.GradientExplainer(wrapped_model, background)

# ============================================================
# 6. EXPLAIN ONLY TOP CLASS (IMPORTANT)
# ============================================================

with torch.no_grad():
    output = wrapped_model(input_tensor)
    pred_class = output.argmax().item()

print("Explaining class:", pred_class)

# Compute SHAP only for that class
shap_values = explainer.shap_values(input_tensor)

# Take only predicted class → reduces load
shap_values = [shap_values[pred_class]]

# ============================================================
# 7. VISUALIZATION (SAFE)
# ============================================================

img_np = input_tensor.squeeze().permute(1,2,0).numpy()

shap.image_plot(shap_values, np.expand_dims(img_np, axis=0))

: 